<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/NCBI_SRA_Data_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Retrieval from NCBI-SRA

## Aim
To retrieve next-generation sequencing (NGS) datasets from the NCBI Sequence Read Archive (SRA).

## Learning Outcomes
By the end of this notebook, students will learn:
- The structure of SRA databases
- How to retrieve metadata for SRA records
- How to download sequencing datasets from SRA
- How to handle and inspect FASTQ files

---

## Background

**NCBI-SRA (Sequence Read Archive)** is a public repository for raw sequencing data (reads) and alignment information from high-throughput sequencing platforms (Illumina, PacBio, Oxford Nanopore, etc.).

### Structure of SRA databases

SRA data is organized in a hierarchy of accession types:

| Level | Accession Prefix | Description |
|---|---|---|
| **Study (Project)** | `SRP` / `PRJNA` | A sequencing study/project, may contain multiple samples |
| **Sample (BioSample)** | `SRS` / `SAMN` | Biological source material used in the study |
| **Experiment** | `SRX` | Describes the sequencing library, platform, and strategy |
| **Run** | `SRR` | The actual sequencing reads (FASTQ/SRA format) generated from an experiment |

So the relationship is roughly:
`Study (SRP) → Sample (SRS) → Experiment (SRX) → Run (SRR)`

One Study can have many Samples; one Sample can have many Experiments; one Experiment can have one or more Runs.

In this notebook, we will:
1. Install the required tools (`sra-tools`, `Biopython`)
2. Query SRA metadata using NCBI **Entrez** utilities
3. Download an example small sequencing run using **prefetch** and **fasterq-dump**
4. Inspect and validate the resulting FASTQ file


## Step 1: Install Required Tools

We need:
- **SRA Toolkit** (`sra-tools`) — official NCBI command-line tools (`prefetch`, `fasterq-dump`, `vdb-validate`) for downloading and converting SRA data.
- **Biopython** — for querying NCBI Entrez metadata (`Bio.Entrez`).

Installing `sra-tools` via `conda`/`bioconda` is the most reliable method on Colab.


In [ ]:
# Install Miniconda-free approach: get precompiled SRA Toolkit binaries directly from NCBI
import os, sys, subprocess

SRA_VERSION = "3.1.1"
url = f"https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VERSION}/sratoolkit.{SRA_VERSION}-ubuntu64.tar.gz"

!wget -q {url} -O sratoolkit.tar.gz
!tar -xzf sratoolkit.tar.gz
!mv sratoolkit.{SRA_VERSION}-ubuntu64 sratoolkit

# Add the toolkit's bin directory to PATH for this session
sra_bin = os.path.abspath("sratoolkit/bin")
os.environ["PATH"] += os.pathsep + sra_bin
print("SRA Toolkit installed at:", sra_bin)


In [ ]:
# Verify installation
!prefetch --version
!fasterq-dump --version


In [ ]:
# Install Biopython for Entrez metadata queries
!pip install -q biopython
import Bio
print("Biopython version:", Bio.__version__)


### One-time SRA Toolkit configuration

The toolkit normally asks for interactive configuration on first run. On Colab we run it non-interactively and accept the defaults so downloads work smoothly.


In [ ]:
# Configure sra-tools non-interactively (accept defaults, enable remote access)
!vdb-config --set /repository/user/main/public/root="$HOME/ncbi_public" 2>/dev/null || true
!vdb-config --cfg-dir="$HOME/.ncbi" --set /LIBS/GUID="00000000-0000-0000-0000-000000000000" 2>/dev/null || true
print("Configuration step complete (safe to ignore warnings above).")


## Step 2: Metadata Retrieval

Before downloading raw reads, it's good practice to inspect the **metadata**: what organism, platform, library strategy, and study the run belongs to.

We can retrieve metadata in two ways:
1. **Programmatically** via `Bio.Entrez` (NCBI's E-utilities API)
2. **Directly** via the SRA Toolkit's `prefetch --info` / the NCBI website

### Example accession used in this notebook
We'll use **`SRR390728`** — a small, well-known public paired-end Illumina RNA-seq run (E. coli), commonly used for teaching because it downloads quickly.


In [ ]:
from Bio import Entrez

# IMPORTANT: Always provide your email when using NCBI Entrez (required by NCBI usage policy)
Entrez.email = "your_email@example.com"   # <-- replace with your own email address

accession = "SRR390728"

# Search the SRA database for this accession to get its internal UID
handle = Entrez.esearch(db="sra", term=accession)
record = Entrez.read(handle)
handle.close()

uid = record["IdList"][0]
print(f"Accession {accession} -> NCBI UID: {uid}")


In [ ]:
# Fetch a human-readable summary (docsum) for this run
handle = Entrez.esummary(db="sra", id=uid)
summary = Entrez.read(handle, validate=False)
handle.close()

# The useful info is inside an XML-like string in 'ExpXml' and 'Runs'
docsum = summary[0]
print("Title:", docsum.get("Title"))
print()
print("--- Experiment metadata (ExpXml) ---")
print(docsum.get("ExpXml"))
print()
print("--- Run metadata (Runs) ---")
print(docsum.get("Runs"))


The XML fields above tell us useful details such as:
- **Organism** sequenced (e.g., *Escherichia coli*)
- **Instrument/platform** used (e.g., Illumina Genome Analyzer)
- **Library strategy** (e.g., RNA-Seq, WGS, ChIP-Seq)
- **Number of spots (reads)** and **total bases**
- The parent **Study (SRP)**, **Sample (SRS)**, and **Experiment (SRX)** accessions

This lets us decide *before downloading* whether a run is relevant and how large it will be.


In [ ]:
# We can also get metadata directly from the SRA Toolkit itself
!prefetch --info SRR390728


## Step 3: Downloading the Sequencing Dataset

We use a two-step process, which is the standard SRA Toolkit workflow:

1. **`prefetch`** — downloads the compact `.sra` file (NCBI's native format) for the run accession.
2. **`fasterq-dump`** — converts the `.sra` file into standard **FASTQ** file(s). For paired-end data, this produces two files (`_1.fastq` and `_2.fastq`).


In [ ]:
# Step 3a: Download the .sra file
!prefetch SRR390728

# Check what was downloaded
!find . -iname "*.sra"


In [ ]:
# Step 3b: Convert the .sra file into FASTQ format
# --split-files separates paired-end reads into two files (_1 and _2)
# -O specifies the output directory
!fasterq-dump SRR390728 --split-files -O ./fastq_output --progress

!ls -lh ./fastq_output


In [ ]:
# Compress the FASTQ files with gzip to save space (common practice for storage/sharing)
!gzip -f ./fastq_output/*.fastq
!ls -lh ./fastq_output


## Step 4: Handling FASTQ Files

A FASTQ file stores each sequencing read as **4 lines**:

```
@READ_ID                     <- Line 1: Sequence identifier
ACTGACTGACTG...               <- Line 2: Nucleotide sequence
+                              <- Line 3: Separator (may repeat the ID)
IIIIIIIIIIII...               <- Line 4: Quality scores (Phred, ASCII-encoded)
```

Let's inspect the downloaded FASTQ file(s) to confirm the download worked and understand their structure.


In [ ]:
import gzip

fastq_file = "./fastq_output/SRR390728_1.fastq.gz"

# Preview the first few reads (first 8 lines = first 2 reads)
with gzip.open(fastq_file, "rt") as f:
    for i in range(8):
        print(f.readline().rstrip())


In [ ]:
# Count the number of reads in the FASTQ file
def count_fastq_reads(path):
    with gzip.open(path, "rt") as f:
        n_lines = sum(1 for _ in f)
    return n_lines // 4

for mate in ["1", "2"]:
    path = f"./fastq_output/SRR390728_{mate}.fastq.gz"
    try:
        n_reads = count_fastq_reads(path)
        print(f"Mate {mate}: {n_reads} reads")
    except FileNotFoundError:
        print(f"Mate {mate}: file not found (run may be single-end)")


In [ ]:
# Basic quality/statistics check using Biopython
from Bio import SeqIO
import gzip

fastq_file = "./fastq_output/SRR390728_1.fastq.gz"

lengths = []
with gzip.open(fastq_file, "rt") as f:
    for i, record in enumerate(SeqIO.parse(f, "fastq")):
        lengths.append(len(record.seq))
        if i == 0:
            print("First read ID:      ", record.id)
            print("First read sequence:", str(record.seq)[:60], "...")
            print("First read quality: ", record.letter_annotations["phred_quality"][:20], "...")
        if i >= 999:   # only scan first 1000 reads for a quick summary
            break

print()
print(f"Scanned {len(lengths)} reads")
print(f"Read length range: {min(lengths)}-{max(lengths)} bp")
print(f"Average read length: {sum(lengths)/len(lengths):.1f} bp")


## Validating the Download (Optional but Recommended)

The SRA Toolkit includes `vdb-validate` to check that a downloaded `.sra` file is not corrupted.


In [ ]:
!vdb-validate SRR390728 2>&1 | tail -20

## Summary

In this experiment, we:
1. Learned the hierarchical structure of SRA (**Study → Sample → Experiment → Run**)
2. Retrieved **metadata** for a run accession (`SRR390728`) using `Bio.Entrez` and `prefetch --info`
3. **Downloaded** the raw sequencing data using `prefetch` and converted it to **FASTQ** using `fasterq-dump`
4. **Inspected** the FASTQ file structure, counted reads, and computed basic read-length statistics
5. **Validated** the integrity of the downloaded data with `vdb-validate`

### Try it yourself
Replace `SRR390728` throughout this notebook with another small run accession (search on the [SRA website](https://www.ncbi.nlm.nih.gov/sra)) and repeat the workflow. Good practice runs to try:
- `SRR000001` (very old, tiny 454 dataset)
- Any small RNA-seq/WGS run you find while browsing a `BioProject` of interest

### Key Commands Reference
| Task | Command |
|---|---|
| Get run info | `prefetch --info <SRR_ID>` |
| Download .sra | `prefetch <SRR_ID>` |
| Convert to FASTQ | `fasterq-dump <SRR_ID> --split-files -O <outdir>` |
| Validate download | `vdb-validate <SRR_ID>` |
